# One-Click Colab Runner

Use this if the whole repo is already in Google Drive.

Manual setup before running:
1. Set Colab runtime to GPU.
2. Put this repo folder in `MyDrive/Neural_Image_Caption_Generator`.
3. Either put Flickr8k in `MyDrive/Neural_Image_Caption_Generator/data/flickr8k`, or set `FLICKR8K_SOURCE_DIR` below to wherever it is in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Change this only if your repo folder has a different name/location.
REPO_DIR = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')

# If Flickr8k is already inside the repo at data/flickr8k, leave this blank.
# If it is somewhere else in Drive, set it, e.g. Path('/content/drive/MyDrive/flickr8k').
FLICKR8K_SOURCE_DIR = None

assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
os.chdir(REPO_DIR)
print('Working in', Path.cwd())

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Prepare Flickr8k

If you do **not** have the dataset in Drive yet, upload your Kaggle `kaggle.json` in the next optional cell and then run the Kaggle download cell.

In [ ]:
import sys, os, csv, random as _random
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

data_root   = REPO_DIR / 'data' / 'flickr8k'
source_dir  = Path(FLICKR8K_SOURCE_DIR).expanduser().resolve() if FLICKR8K_SOURCE_DIR else (REPO_DIR / 'data' / 'flickr8k').resolve()
data_root.mkdir(parents=True, exist_ok=True)

# ── 1. images folder ────────────────────────────────────────────────────────
image_dir = None
for _c in source_dir.rglob('*'):
    if _c.is_dir() and _c.name.lower() in {'images', 'flickr8k_dataset', 'flicker8k_dataset'}:
        if len(list(_c.glob('*.jpg'))) > 100:
            image_dir = _c
            break
assert image_dir, f'No images folder found under {source_dir}'
img_dst = data_root / 'images'
if not img_dst.exists() and not img_dst.is_symlink():
    os.symlink(image_dir.resolve(), img_dst)
    print(f'Linked images folder: {image_dir}')

# ── 2. captions.txt ─────────────────────────────────────────────────────────
caps_src = next(source_dir.rglob('captions.txt'), None)
assert caps_src, f'captions.txt not found under {source_dir}'
caps_dst = data_root / 'captions.txt'
if not caps_dst.exists() and not caps_dst.is_symlink():
    os.symlink(caps_src.resolve(), caps_dst)
    print('Linked captions.txt')

# ── 3. Split files – link if present, otherwise auto-generate ───────────────
SPLIT_NAMES = ['Flickr_8k.trainImages.txt', 'Flickr_8k.devImages.txt', 'Flickr_8k.testImages.txt']
for _fname in SPLIT_NAMES:
    _src = next(source_dir.rglob(_fname), None)
    if _src:
        _dst = data_root / _fname
        if not _dst.exists() and not _dst.is_symlink():
            os.symlink(_src.resolve(), _dst)

_splits_generated = False
if any(not (data_root / f).exists() for f in SPLIT_NAMES):
    print('Split files missing – generating from captions.txt ...')
    _names = set()
    with open(caps_dst, newline='') as _f:
        for _raw in _f:
            _line = _raw.strip()
            if not _line: continue
            if '\t' in _line:
                _img_id = _line.split('\t', 1)[0]
            else:
                _row = next(csv.reader([_line]))
                if not _row or _row[0].lower() in {'image', 'image_name', 'filename'}: continue
                _img_id = _row[0]
            _names.add(os.path.basename(_img_id.strip().split('#')[0]))
    _names = sorted(_names)
    _rng = _random.Random(42)
    _rng.shuffle(_names)
    _n = len(_names)
    print(f'  {_n} unique images found')
    # Fixed val/test pools; ALL remaining images go to training.
    _n_val   = min(1000, max(1, _n // 8))
    _n_test  = min(1000, max(1, _n // 8))
    _n_train = _n - _n_val - _n_test
    for _fname, _imgs in [
        ('Flickr_8k.trainImages.txt', _names[:_n_train]),
        ('Flickr_8k.devImages.txt',   _names[_n_train:_n_train+_n_val]),
        ('Flickr_8k.testImages.txt',  _names[_n_train+_n_val:_n_train+_n_val+_n_test]),
    ]:
        _dst = data_root / _fname
        if not _dst.exists():
            _dst.write_text('\n'.join(_imgs) + '\n')
            print(f'  Generated {_fname} ({len(_imgs)} images)')
    _splits_generated = True

# ── 4. Validate (skip strict count check when we generated splits) ───────────
from utils import validate_dataset_layout
validate_dataset_layout(str(data_root), strict_split_counts=not _splits_generated)
print('Dataset ready:', data_root)


### Optional Kaggle Download Path

Only use these two cells if Flickr8k is not already in Drive. In Kaggle, create an API token and upload `kaggle.json` here.

In [ ]:
# OPTIONAL: upload kaggle.json, then run the next cell.
# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# OPTIONAL: Kaggle download. Uncomment only if you used the kaggle.json cell above.
# !python scripts/colab_setup.py --repo_dir "$REPO_DIR" --download_kaggle

## Validate Code

In [ ]:
!pytest -q

## Train

In [ ]:
!python train.py

## Evaluate

In [ ]:
!python evaluate.py \
  --checkpoint checkpoints/best.pt \
  --data_root data/flickr8k \
  --vocab data/flickr8k/vocab.json \
  --split test \
  --batch_size 64 \
  --results_out results/test_bleu.json

## Visualize Attention

In [ ]:
!python visualize.py \
  --checkpoint checkpoints/best.pt \
  --vocab data/flickr8k/vocab.json \
  --data_root data/flickr8k \
  --split test \
  --num_examples 6 \
  --output_dir results/attention_examples \
  --overlay_style paper \
  --dpi 200

In [ ]:
from IPython.display import Image, display
for path in sorted(Path('results/attention_examples').glob('*.png'))[:6]:
    print(path)
    display(Image(filename=str(path)))

## Ablation Experiments

Five controlled experiments testing which components of the soft-attention model matter.  
Outputs are saved to `results/ablation_results/<name>/` inside this Drive folder and persist automatically.

| Experiment | What it removes |
|---|---|
| `baseline_soft_attention` | Nothing — paper's exact model |
| `no_attention_mean` | Dynamic attention → static mean context |
| `no_doubly_stochastic_lambda_0` | Doubly stochastic penalty (Eq. 14, λ=0) |
| `no_beta_gate` | Learned β scalar gate (forced to 1) |
| `feature_grid_7x7` | 14×14 spatial grid → 7×7 (L=196 → 49) |

**Each training run takes as long as the baseline.** Run one cell block at a time.  
The output folder is created automatically — no need to make it in Drive first.

### Preview all commands (no training started)

In [ ]:
!python scripts/run_ablations.py

### a) Baseline Soft Attention

Paper's exact model with all default flags. Run this first — it's the comparison anchor.

In [ ]:
!python train.py \
  --checkpoint_dir results/ablation_results/baseline_soft_attention/checkpoints \
  --results_dir    results/ablation_results/baseline_soft_attention

In [ ]:
!python evaluate.py \
  --checkpoint  results/ablation_results/baseline_soft_attention/checkpoints/best.pt \
  --data_root   data/flickr8k \
  --vocab       data/flickr8k/vocab.json \
  --split       test \
  --results_out results/ablation_results/baseline_soft_attention/test_bleu.json

### b) No Attention — Static Mean Context

Replaces the dynamic context vector with the static mean-pooled image feature at every step.  
Tests whether *any* form of dynamic attention helps over averaging all spatial locations.

In [ ]:
!python train.py \
  --attention_mode none \
  --checkpoint_dir results/ablation_results/no_attention_mean/checkpoints \
  --results_dir    results/ablation_results/no_attention_mean

In [ ]:
!python evaluate.py \
  --attention_mode none \
  --checkpoint  results/ablation_results/no_attention_mean/checkpoints/best.pt \
  --data_root   data/flickr8k \
  --vocab       data/flickr8k/vocab.json \
  --split       test \
  --results_out results/ablation_results/no_attention_mean/test_bleu.json

### c) No Doubly Stochastic Regularisation (λ=0)

Sets λ=0, fully disabling the Eq. 14 penalty.  
The paper claims this penalty "was important quantitatively to improving overall BLEU score".

In [ ]:
!python train.py \
  --lambda_weight 0.0 \
  --checkpoint_dir results/ablation_results/no_doubly_stochastic_lambda_0/checkpoints \
  --results_dir    results/ablation_results/no_doubly_stochastic_lambda_0

In [ ]:
!python evaluate.py \
  --checkpoint  results/ablation_results/no_doubly_stochastic_lambda_0/checkpoints/best.pt \
  --data_root   data/flickr8k \
  --vocab       data/flickr8k/vocab.json \
  --split       test \
  --results_out results/ablation_results/no_doubly_stochastic_lambda_0/test_bleu.json

### d) No Beta Gate

Forces β=1 at every step, removing the learned scalar gate on the context vector (paper Sec. 4.2.1).

In [ ]:
!python train.py \
  --no_beta_gate \
  --checkpoint_dir results/ablation_results/no_beta_gate/checkpoints \
  --results_dir    results/ablation_results/no_beta_gate

In [ ]:
!python evaluate.py \
  --no_beta_gate \
  --checkpoint  results/ablation_results/no_beta_gate/checkpoints/best.pt \
  --data_root   data/flickr8k \
  --vocab       data/flickr8k/vocab.json \
  --split       test \
  --results_out results/ablation_results/no_beta_gate/test_bleu.json

### e) Feature Grid 7×7 (L=49)

Uses a 7×7 spatial feature map (L=49) via adaptive average pooling instead of 14×14 (L=196).

In [ ]:
!python train.py \
  --feature_grid_size 7 \
  --checkpoint_dir results/ablation_results/feature_grid_7x7/checkpoints \
  --results_dir    results/ablation_results/feature_grid_7x7

In [ ]:
!python evaluate.py \
  --feature_grid_size 7 \
  --checkpoint  results/ablation_results/feature_grid_7x7/checkpoints/best.pt \
  --data_root   data/flickr8k \
  --vocab       data/flickr8k/vocab.json \
  --split       test \
  --results_out results/ablation_results/feature_grid_7x7/test_bleu.json

### Results Summary

Run after any experiments finish. Missing experiments are skipped automatically.

In [ ]:
import json
from pathlib import Path

BASE = 'results/ablation_results'
EXPERIMENTS = [
    ('baseline_soft_attention',       'Baseline soft attention (paper)'),
    ('no_attention_mean',              'No attention — static mean context'),
    ('no_doubly_stochastic_lambda_0',  'No doubly stochastic reg (λ=0)'),
    ('no_beta_gate',                   'No beta gate (β forced to 1)'),
    ('feature_grid_7x7',               'Feature grid 7×7 (L=49)'),
]
PAPER = dict(bleu1=67.0, bleu2=44.8, bleu3=29.9, bleu4=19.5, meteor=18.93)

rows = []
for name, label in EXPERIMENTS:
    path = Path(f'{BASE}/{name}/test_bleu.json')
    if not path.exists():
        rows.append((label, None))
        continue
    with open(path) as f:
        s = json.load(f)['scores']
    rows.append((label, s))

hdr = f"{'Experiment':<42} {'B-1':>5} {'B-2':>5} {'B-3':>5} {'B-4':>5} {'METEOR':>7}"
sep = '-' * len(hdr)
print(hdr); print(sep)
print(f"{'Paper target (soft attention)':<42} {PAPER['bleu1']:>5.1f} {PAPER['bleu2']:>5.1f} {PAPER['bleu3']:>5.1f} {PAPER['bleu4']:>5.1f} {PAPER['meteor']:>7.2f}")
print(sep)
for label, s in rows:
    if s is None:
        print(f"{label:<42} {'(not run yet)':>30}")
    else:
        m = s.get('meteor', float('nan'))
        print(f"{label:<42} {s['bleu1']*100:>5.1f} {s['bleu2']*100:>5.1f} {s['bleu3']*100:>5.1f} {s['bleu4']*100:>5.1f} {m*100:>7.2f}")
